# Spark DataFrame Fundamentals

This notebook introduces structured data with Apache Spark 3.5.9. You will create a DataFrame, inspect its schema, and use common projection, filtering, column, sorting, and null-handling operations.

## Learning objectives

- Explain the relationship between rows, columns, and a schema.
- Distinguish lazy transformations from actions.
- Select, derive, rename, drop, filter, and sort columns.
- Replace null values safely.

## 1. Start Spark before running the notebook

This lesson uses the single-node Spark **standalone cluster**, not `local` mode. In a WSL terminal, start one master and one worker:

```bash
/opt/spark/sbin/start-master.sh
/opt/spark/sbin/start-worker.sh "spark://$(hostname):7077"
jps
```

Open the master UI at [http://localhost:8080](http://localhost:8080). It should show one live worker. The notebook derives the master URL from the WSL hostname. If your lab uses a different URL, set `SPARK_MASTER` before starting Jupyter.

In [ ]:
import os
import socket

from pyspark.sql import SparkSession

master_url = os.environ.get(
    "SPARK_MASTER",
    f"spark://{socket.gethostname()}:7077",
)

spark = (
    SparkSession.builder
    .appName("D30-DataFrame-Basics")
    .master(master_url)
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Spark version :", spark.version)
print("Spark master  :", sc.master)
print("Application ID:", sc.applicationId)
print("Spark UI      :", sc.uiWebUrl)

## 2. Create a DataFrame

A DataFrame is a distributed table with named, typed columns. Spark uses the schema to validate expressions and build logical and physical execution plans. Transformations return new immutable DataFrames; actions trigger execution.

For teaching convenience, Spark infers the types in this small in-memory dataset. For files and production pipelines, prefer an explicit schema.

In [ ]:
products = [
    (1, "iPhone", 1000.00, 100, 0.0),
    (2, "Galaxy", 545.50, 101, None),
    (3, "Pixel", 645.99, 102, None),
]

columns = ["product_id", "product_name", "price", "brand_id", "offer"]
product_df = spark.createDataFrame(products, columns)

product_df.printSchema()
product_df.show()

## 3. Inspect rows and partitions

`collect()` returns every row to the driver, so use it only when the result is small. A DataFrame also exposes an underlying RDD of `Row` objects, but normal structured processing should use the DataFrame API so Spark can optimize it.

In [ ]:
print("Partitions:", product_df.rdd.getNumPartitions())
print("First two rows:", product_df.take(2))

## 4. Transformations are lazy; actions run jobs

`filter` creates a new DataFrame and records the operation. It does not mutate `product_df`. `show` is the action that asks Spark to execute the plan.

In [ ]:
affordable_df = product_df.filter(product_df["price"] <= 750)
affordable_df.explain(mode="simple")
affordable_df.show()

## 5. Select and derive columns

`select` projects columns. `selectExpr` accepts SQL expressions, while `withColumn` uses the Python Column API. Aliases give derived expressions meaningful output names.

In [ ]:
from pyspark.sql.functions import col, lit, upper

product_df.select("product_name", "price").show()

product_df.select(
    upper("product_name").alias("product_name_upper"),
    "price",
    (col("price") * 0.90).alias("discounted_price"),
).show()

product_df.selectExpr(
    "product_name",
    "price",
    "price * 0.90 AS discounted_price",
).show()

`lit` creates a constant-valued column. The second `withColumn` can refer to `quantity` because the first call has already added it to the new DataFrame.

In [ ]:
order_df = (
    product_df
    .withColumn("quantity", lit(4))
    .withColumn("amount", col("quantity") * col("price"))
)

order_df.printSchema()
order_df.show()

## 6. Rename and drop columns

These operations return new DataFrames. The original `product_df` remains unchanged.

In [ ]:
renamed_df = product_df.withColumnRenamed("price", "unit_price")
reduced_df = renamed_df.drop("brand_id")

reduced_df.printSchema()
reduced_df.show()

## 7. Filter rows

`filter` and `where` are aliases. Conditions can use Column expressions or SQL expression strings. When combining Column conditions, wrap each comparison in parentheses and use `&` for AND or `|` for OR.

In [ ]:
product_df.filter(
    (col("price") >= 500) & (col("price") < 600)
).show()

product_df.where("price >= 500 AND price < 600").show()

## 8. Sort rows

The default order is ascending. Use `desc()` or a column's `.desc()` method for descending order.

In [ ]:
product_df.orderBy("price").show()
product_df.orderBy(col("price").desc()).show()

## 9. Handle null values

`fillna` creates a new DataFrame. Limit replacement to appropriate columns so one default is not accidentally applied across unrelated fields.

In [ ]:
filled_df = product_df.fillna({"offer": 0.0})
filled_df.show()

## 10. Practice

Create a result containing `product_name`, `price`, and a `price_with_tax` column calculated at 18%. Keep products priced above 600, sort by the new value in descending order, and display the result.

In [ ]:
# Write the practice solution here.

## 11. Stop Spark

Run this cell when the lesson is complete.

In [ ]:
spark.stop()
print("Spark session stopped.")